# Train the Waterbirds ERM base model used by Evidential Alignment

This notebook trains the **base ERM model** that `evidential_alignment.py` expects through `--erm_model`.

The model is:

\[
\text{ImageNet-pretrained ResNet-50 backbone}
\;+\;
\text{fresh Linear(2048, 2) classifier}
\]

and the **entire network is fine-tuned end-to-end** on the Waterbirds **training split** for 100 epochs with ordinary cross-entropy.

Repository-default ERM optimization used here:

- optimizer: SGD
- learning rate: `1e-3`
- momentum: `0.9`
- weight decay: `1e-4`
- epochs: `100`
- batch size: `128`
- seed: `0`
- pretrained backbone: `True`

The repository evaluates on the Waterbirds validation set every epoch and saves `best_val_acc_model.pt`. That full checkpoint is the one this notebook preserves permanently on Google Drive for later use by Evidential Alignment.

For extra safety, this notebook also sets `--save_freq 10`. This **does not change training**; it only writes recovery checkpoints every 10 epochs.

In [ ]:
# ============================================================
# 1. MOUNT DRIVE AND DEFINE PERSISTENT PATHS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
from datetime import datetime

# Shared project folder on Drive. Renamed to "FeatureRater" (Nov 2026); a Drive shortcut
# may still carry the old name, so both are accepted (the folder must contain archives/).
DRIVE_ROOT_NAMES = ("FeatureRater", "evidential_alignment_colab")
DRIVE_ROOT = next(
    (Path("/content/drive/MyDrive") / name for name in DRIVE_ROOT_NAMES
     if (Path("/content/drive/MyDrive") / name / "archives").exists()),
    Path("/content/drive/MyDrive") / DRIVE_ROOT_NAMES[-1],
)
print("Project folder on Drive:", DRIVE_ROOT)

ARCHIVE_DIR = (
    DRIVE_ROOT / "archives"
)

# Drive layout (Nov 2026): one folder per ERM backbone, waterbirds/<ERM_ID>/.
# The original checkpoint keeps working from its old place until it is moved.
ERM_ID = "erm-r50-e100-s0"
ERM_SAVE_ROOT = DRIVE_ROOT / "waterbirds" / ERM_ID
_RUN = "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained"
_OLD_ROOT = DRIVE_ROOT / "erm_base_models"
if (
    ERM_ID == "erm-r50-e100-s0"
    and not (ERM_SAVE_ROOT / _RUN / "best_val_acc_model.pt").exists()
    and (_OLD_ROOT / _RUN / "best_val_acc_model.pt").exists()
):
    ERM_SAVE_ROOT = _OLD_ROOT

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ERM_SAVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPO_DIR = Path(
    "/content/evidential_alignment"
)

LOCAL_DATA_ROOT = Path(
    "/content/datasets"
)

WB_DIR = (
    LOCAL_DATA_ROOT
    / "waterbird_complete95_forest2water2"
)

WB_ARCHIVE = (
    ARCHIVE_DIR
    / "waterbird_complete95_forest2water2.tar.gz"
)

print("Drive root:")
print(DRIVE_ROOT)

print("\nERM models will be saved permanently under:")
print(ERM_SAVE_ROOT)

In [ ]:
# ============================================================
# 2. CLONE THE EVIDENTIAL ALIGNMENT REPOSITORY
# ============================================================

REPO_URL = (
    "https://github.com/"
    "TheRealMmd/evidential_alignment.git"
)

if not REPO_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

    print("✓ Repository cloned.")

else:

    print("✓ Repository already exists:")
    print(REPO_DIR)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm",
        "transformers",
        "wilds",
        "wget",
    ],
    check=True,
)

print("✓ Dependencies ready.")

In [ ]:
# ============================================================
# 3. PREPARE WATERBIRDS FROM YOUR PERSISTENT DRIVE ARCHIVE
# ============================================================

LOCAL_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WB_URL = (
    "https://downloads.cs.stanford.edu/"
    "nlp/data/dro/"
    "waterbird_complete95_forest2water2.tar.gz"
)

if not (
    WB_ARCHIVE.exists()
    and WB_ARCHIVE.stat().st_size > 100_000_000
):

    print(
        "Persistent Waterbirds archive not found; "
        "downloading it once to Drive..."
    )

    subprocess.run(
        [
            "wget",
            "-c",
            WB_URL,
            "-O",
            str(WB_ARCHIVE),
        ],
        check=True,
    )

else:

    print("✓ Reusing Waterbirds archive from Drive:")
    print(WB_ARCHIVE)


if not (
    WB_DIR / "metadata.csv"
).exists():

    print("\nExtracting Waterbirds into local Colab storage...")

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(WB_ARCHIVE),
            "-C",
            str(LOCAL_DATA_ROOT),
        ],
        check=True,
    )

else:

    print("✓ Waterbirds is already extracted locally.")


assert (
    WB_DIR / "metadata.csv"
).exists(), (
    f"Waterbirds metadata missing at {WB_DIR}"
)

print("\n✓ Waterbirds ready:")
print(WB_DIR)

✓ Reusing Waterbirds archive from Drive:
/content/drive/MyDrive/evidential_alignment_colab/archives/waterbird_complete95_forest2water2.tar.gz

Extracting Waterbirds into local Colab storage...

✓ Waterbirds ready:
/content/datasets/waterbird_complete95_forest2water2


In [ ]:
# ============================================================
# 4. PATCH REPOSITORY DATASET PATH
# ============================================================

config_path = (
    REPO_DIR / "config.py"
)

config_text = (
    config_path.read_text(
        encoding="utf-8"
    )
)

config_text = config_text.replace(
    '"waterbirds": "/path/to/datasets/waterbirds"',
    f'"waterbirds": "{WB_DIR}"',
)

config_path.write_text(
    config_text,
    encoding="utf-8",
)

print("✓ config.py Waterbirds path:")
print(WB_DIR)

✓ config.py Waterbirds path:
/content/datasets/waterbird_complete95_forest2water2


In [ ]:
# ============================================================
# 5. PYTORCH 2.6+ CHECKPOINT-LOADING FIX
#
# This does NOT change training.
# It only allows the old repository to reload checkpoints that
# contain its config/optimizer objects after training.
# ============================================================

algorithm_path = (
    REPO_DIR
    / "algorithms"
    / "algorithm.py"
)

text = algorithm_path.read_text(
    encoding="utf-8"
)

old_double = (
    'torch.load(check_point, map_location="cpu")'
)

old_single = (
    "torch.load(check_point, map_location='cpu')"
)

new_load = (
    'torch.load('
    'check_point, '
    'map_location="cpu", '
    'weights_only=False'
    ')'
)

changed = False

if old_double in text:
    text = text.replace(
        old_double,
        new_load,
    )
    changed = True

if old_single in text:
    text = text.replace(
        old_single,
        new_load,
    )
    changed = True

algorithm_path.write_text(
    text,
    encoding="utf-8",
)

if changed:
    print(
        "✓ Patched repository checkpoint loading "
        "for PyTorch 2.6+."
    )
else:
    print(
        "✓ No old torch.load pattern found; "
        "the repository may already be patched."
    )

✓ Patched repository checkpoint loading for PyTorch 2.6+.


In [ ]:
# ============================================================
# 6. TRAINING CONFIGURATION AND EXPECTED DRIVE PATHS
# ============================================================

SEED = 0

BATCH_SIZE = 128
NUM_WORKERS = 2

EPOCHS = 100

LR = 1e-3
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# Saving every 10 epochs is only for recovery.
# It does not change the optimization.
SAVE_FREQ = 10

FORCE_RETRAIN = False


EXPERIMENT_DIR = (
    ERM_SAVE_ROOT
    / (
        "erm_waterbirds_resnet50_"
        "train_128B_100E_seed0_pretrained"
    )
)

BEST_VAL_ACC_CHECKPOINT = (
    EXPERIMENT_DIR
    / "best_val_acc_model.pt"
)

LATEST_CHECKPOINT = (
    EXPERIMENT_DIR
    / "latest_model.pt"
)

STABLE_ERM_CHECKPOINT = (
    ERM_SAVE_ROOT
    / (
        "waterbirds_erm_resnet50_"
        "imagenet_100ep_best_val_acc.pt"
    )
)

RESULT_PATH = (
    ERM_SAVE_ROOT
    / "waterbirds_erm_training_results.txt"
)


print("=" * 100)
print("ERM BASE MODEL CONFIGURATION")
print("=" * 100)

print("Backbone:         ResNet-50")
print("Initialization:   ImageNet pretrained")
print("Training:         END-TO-END backbone + Linear(2048 -> 2)")
print("Dataset:          Waterbirds train")
print("Loss:             ordinary cross entropy")
print(f"Epochs:           {EPOCHS}")
print(f"Batch size:       {BATCH_SIZE}")
print(f"Learning rate:    {LR}")
print(f"Momentum:         {MOMENTUM}")
print(f"Weight decay:     {WEIGHT_DECAY}")
print(f"Seed:             {SEED}")

print()
print("Repository experiment directory:")
print(EXPERIMENT_DIR)

print()
print("Expected best-val-accuracy checkpoint:")
print(BEST_VAL_ACC_CHECKPOINT)

print()
print("Stable permanent EA checkpoint:")
print(STABLE_ERM_CHECKPOINT)

print("=" * 100)

ERM BASE MODEL CONFIGURATION
Backbone:         ResNet-50
Initialization:   ImageNet pretrained
Training:         END-TO-END backbone + Linear(2048 -> 2)
Dataset:          Waterbirds train
Loss:             ordinary cross entropy
Epochs:           100
Batch size:       128
Learning rate:    0.001
Momentum:         0.9
Weight decay:     0.0001
Seed:             0

Repository experiment directory:
/content/drive/MyDrive/evidential_alignment_colab/erm_base_models/erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained

Expected best-val-accuracy checkpoint:
/content/drive/MyDrive/evidential_alignment_colab/erm_base_models/erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained/best_val_acc_model.pt

Stable permanent EA checkpoint:
/content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt


In [ ]:
# ============================================================
# 7. TRAIN ERM
#
# If the finished checkpoint already exists on Drive, this cell
# reuses it unless FORCE_RETRAIN=True.
# ============================================================

import subprocess
import sys
import os

already_finished = (
    BEST_VAL_ACC_CHECKPOINT.exists()
    and LATEST_CHECKPOINT.exists()
)

if already_finished and not FORCE_RETRAIN:

    print("=" * 100)
    print("FINISHED ERM CHECKPOINT ALREADY EXISTS")
    print("=" * 100)

    print(
        "Skipping the 100-epoch training run."
    )

    print(
        BEST_VAL_ACC_CHECKPOINT
    )

else:

    print("=" * 100)
    print("STARTING 100-EPOCH WATERBIRDS ERM TRAINING")
    print("=" * 100)

    cmd = [
        sys.executable,
        "-u",
        "main.py",

        "--dataset",
        "waterbirds",

        "--save_folder",
        str(
            ERM_SAVE_ROOT
        ),

        "--algorithm",
        "erm",

        "--backbone",
        "resnet50",

        "--pretrained",
        "True",

        "--batch_size",
        str(
            BATCH_SIZE
        ),

        "--num_workers",
        str(
            NUM_WORKERS
        ),

        "--mode",
        "train",

        "--train_split",
        "train",

        "--test_split",
        "test",

        "--split_train",
        "1.0",

        "--split_val",
        "1.0",

        "--optimizer",
        "sgd",

        "--optimizer_kwargs",
        f"lr={LR}",
        f"momentum={MOMENTUM}",
        f"weight_decay={WEIGHT_DECAY}",

        "--scheduler",
        "none",

        "--epoch",
        str(
            EPOCHS
        ),

        "--eval_freq",
        "1",

        "--save_freq",
        str(
            SAVE_FREQ
        ),

        "--gpu",
        "0",

        "--seed",
        str(
            SEED
        ),

        "--result_path",
        str(
            RESULT_PATH
        ),
    ]


    print("\nCOMMAND:\n")
    print(
        " ".join(
            cmd
        )
    )

    print("\n" + "=" * 100)


    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"


    process = subprocess.Popen(
        cmd,
        cwd=str(
            REPO_DIR
        ),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )


    try:

        for line in iter(
            process.stdout.readline,
            "",
        ):

            print(
                line,
                end="",
                flush=True,
            )

    finally:

        if process.stdout is not None:
            process.stdout.close()


    return_code = (
        process.wait()
    )

    print("\n" + "=" * 100)
    print(
        "ERM subprocess return code:",
        return_code,
    )


    if BEST_VAL_ACC_CHECKPOINT.exists():

        print(
            "✓ best_val_acc_model.pt exists on Drive."
        )

    else:

        print(
            "\nAvailable checkpoint files under experiment directory:"
        )

        if EXPERIMENT_DIR.exists():

            for path in sorted(
                EXPERIMENT_DIR.glob(
                    "*.pt"
                )
            ):
                print(path)

        raise RuntimeError(
            "Training ended without producing "
            "best_val_acc_model.pt."
        )


    if return_code != 0:

        print(
            "\nWARNING: subprocess returned non-zero, "
            "but the best ERM checkpoint exists."
        )

        print(
            "The checkpoint will still be verified and "
            "copied to its permanent alias in the next cell."
        )

    else:

        print(
            "✓ ERM training/testing completed normally."
        )

In [ ]:
# ============================================================
# 8. VERIFY AND SAVE A STABLE PERMANENT CHECKPOINT
# ============================================================

import torch
import shutil
import json

assert BEST_VAL_ACC_CHECKPOINT.exists(), (
    "best_val_acc_model.pt is missing:\n"
    f"{BEST_VAL_ACC_CHECKPOINT}"
)


checkpoint = torch.load(
    BEST_VAL_ACC_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


required_keys = {
    "model_sd",
    "sel_metric",
    "config",
    "epoch",
}

missing = (
    required_keys
    - set(
        checkpoint.keys()
    )
)

assert not missing, (
    f"Checkpoint is missing keys: {missing}"
)


print("=" * 100)
print("BEST ERM CHECKPOINT VERIFIED")
print("=" * 100)

print("Source:")
print(BEST_VAL_ACC_CHECKPOINT)

print()
print(
    "Best checkpoint was saved at training epoch:",
    checkpoint[
        "epoch"
    ],
)

print(
    "Validation accuracy associated with checkpoint:",
    checkpoint[
        "sel_metric"
    ],
)

print(
    "Number of tensors in model_sd:",
    len(
        checkpoint[
            "model_sd"
        ]
    ),
)


shutil.copy2(
    BEST_VAL_ACC_CHECKPOINT,
    STABLE_ERM_CHECKPOINT,
)


stable = torch.load(
    STABLE_ERM_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert (
    "model_sd"
    in stable
)


manifest = {
    "dataset": "waterbirds",
    "algorithm": "erm",
    "backbone": "resnet50",
    "pretrained": True,
    "pretraining_source": "ImageNet",
    "training_scope": "full backbone + classifier",
    "train_split": "train",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "sgd",
    "learning_rate": LR,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "repository_checkpoint": str(
        BEST_VAL_ACC_CHECKPOINT
    ),
    "stable_checkpoint": str(
        STABLE_ERM_CHECKPOINT
    ),
    "best_checkpoint_epoch": int(
        stable[
            "epoch"
        ]
    ),
    "best_val_accuracy": float(
        stable[
            "sel_metric"
        ]
    ),
}

MANIFEST_PATH = (
    ERM_SAVE_ROOT
    / "waterbirds_erm_resnet50_imagenet_100ep_manifest.json"
)

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)


print()
print("=" * 100)
print("SUCCESS — PERMANENT ERM MODEL SAVED ON DRIVE")
print("=" * 100)

print()
print(
    "Use this checkpoint for future Evidential Alignment runs:"
)

print(
    STABLE_ERM_CHECKPOINT
)

print()
print(
    f"Size: "
    f"{STABLE_ERM_CHECKPOINT.stat().st_size / (1024**2):.2f} MB"
)

print()
print("Manifest:")
print(MANIFEST_PATH)

print("=" * 100)


ERM_MODEL_PATH = (
    STABLE_ERM_CHECKPOINT
)

## Which saved model should I use later?

Use:

```text
<project folder>/waterbirds/erm-r50-e100-s0/
waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
```

This is a **full repository checkpoint** containing the ResNet-50 backbone, classifier head, saved configuration, optimizer state, selection metric, and checkpoint epoch.

For Evidential Alignment, pass that file to:

```bash
--erm_model <PATH>
```

Do **not** use one of the embedding-cache `.pt` files as `--erm_model`; those are only stored feature tensors, not a trained ResNet-50 classifier checkpoint.

In [ ]:
# ============================================================
# 9. EXAMPLE: PATH FOR YOUR FUTURE EVIDENTIAL ALIGNMENT RUN
# ============================================================

from pathlib import Path

ERM_MODEL_PATH = STABLE_ERM_CHECKPOINT

assert ERM_MODEL_PATH.exists()

print(
    "Future Evidential Alignment --erm_model path:"
)

print(
    ERM_MODEL_PATH
)

print()
print(
    "Example command fragment:"
)

print(
    f"--erm_model {ERM_MODEL_PATH}"
)